# MotifForge · Fine-tune Qwen3 to draw SVGs

Train Qwen3-1.7B on captioned SVGs and compare it with the base model. In Colab, select **Runtime → Change runtime type → T4 GPU**. Run the cells in order and inspect the dataset preview and run estimate before starting training.

A free Colab session can disconnect. To continue, reconnect to a GPU, reopen this notebook, keep the same run name, set the action to **resume**, and run from the top. Progress is saved in your Google Drive.

## 1. Check the GPU and install the trainer

In [ ]:
import subprocess
import sys

gpu_check = """
import torch

if not torch.cuda.is_available():
    raise SystemExit(f'PyTorch {torch.__version__} cannot access CUDA.')
print(f'GPU: {torch.cuda.get_device_name(0)} | PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
"""
try:
    subprocess.run([sys.executable, '-c', gpu_check], check=True, timeout=45)
except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
    raise RuntimeError('In Colab, select Runtime > Change runtime type > T4 GPU, reconnect, and rerun this cell.') from exc

In [ ]:
# Check again before starting package installation.
subprocess.run([sys.executable, '-c', gpu_check], check=True, timeout=45)

pip = [sys.executable, '-m', 'pip', 'install', '--retries', '1', '--timeout', '20']
subprocess.run([*pip, 'sentencepiece', 'protobuf', 'datasets==4.3.0',
                'huggingface_hub>=0.34.0', 'hf_transfer', 'pyyaml', 'cairosvg', 'scikit-image'], check=True)
subprocess.run([*pip, '--no-deps', 'unsloth_zoo', 'bitsandbytes', 'accelerate',
                'peft', 'triton', 'unsloth'], check=True)
subprocess.run(['unsloth', 'install-kernels'], check=True)
subprocess.run([*pip, '--no-deps', '--upgrade', 'torchao>=0.16.0'], check=True)
subprocess.run([*pip, 'transformers==4.57.6'], check=True)
subprocess.run([*pip, '--no-deps', 'trl==0.22.2'], check=True)
subprocess.run([sys.executable, '-c',
                'from unsloth import FastLanguageModel; from trl import SFTConfig, SFTTrainer'],
               check=True)
print('Unsloth and TRL are ready.')

## 2. Name the run and mount Drive

For a new run, check [Google Drive storage](https://drive.google.com/drive/quota) and confirm that at least **3 GB** is free. The run keeps two rolling training checkpoints, its prepared data, adapter, and evidence. Model downloads, temporary checkpoints, and export files stay in the Colab session. Use a unique name for each new run and only one active Colab session per run.

In [ ]:
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import re
import shutil

from google.colab import drive

RUN_NAME = 'motifforge-01' # @param {type:"string"}
RUN_ACTION = 'start' # @param ["start", "resume"]
DRIVE_SPACE_CONFIRMED = False # @param {type:"boolean"}

if RUN_ACTION not in {'start', 'resume'} or not re.fullmatch(r'[a-zA-Z0-9][a-zA-Z0-9_-]{0,63}', RUN_NAME):
    raise ValueError('Choose start or resume and enter a run name with letters, numbers, _ or -.')
if RUN_ACTION == 'start' and not DRIVE_SPACE_CONFIRMED:
    raise ValueError('Check that Drive has 3 GB free, then tick DRIVE_SPACE_CONFIRMED.')
drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/MotifForge') / RUN_NAME
WORK = Path('/content/motifforge-work') / RUN_NAME
if RUN_ACTION == 'start':
    if RUN.exists():
        raise FileExistsError('That run already exists. Choose resume or enter a new run name.')
    RUN.mkdir(parents=True)
elif not RUN.is_dir():
    raise FileNotFoundError('No saved run has that name. Check the name or choose start.')
WORK.mkdir(parents=True, exist_ok=True)
print('Drive run folder:', RUN)

def save_json(path, value):
    temporary = path.with_suffix(path.suffix + '.partial')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporary.replace(path)

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def digest(path):
    with path.open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

def now():
    return datetime.now(timezone.utc).isoformat()

def git(*args):
    return subprocess.check_output(['git', *map(str, args)], text=True).strip()

ROOT = WORK / 'repo'
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--branch', 'motifforge-colab',
                    'https://github.com/Maaz-319/finetune-anywhere.git', str(ROOT)], check=True)
source_lock = RUN / 'repo.json'
if source_lock.exists():
    repo_revision = read_json(source_lock)['revision']
    git('-C', ROOT, 'checkout', '--detach', repo_revision)
else:
    repo_revision = git('-C', ROOT, 'rev-parse', 'HEAD')
    save_json(source_lock, {'revision': repo_revision})
sys.path[:0] = [str(ROOT / 'motifforge'), str(ROOT / 'scripts')]
print('Project revision:', repo_revision)

## 3. Prepare and review the dataset

The notebook streams the [captioned subset](https://huggingface.co/datasets/JackertheHacker/svg-stack-qwen-captioned-subset) derived from [StarVector SVG-Stack](https://huggingface.co/datasets/starvector/svg-stack). The prepared splits are saved to Drive. Review the sample captions and images before moving on.

In [ ]:
from collections import Counter
from importlib.metadata import version
import xml.etree.ElementTree as ET

# Import Unsloth before Transformers or TRL.
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
import yaml
from datasets import Dataset, load_dataset
from huggingface_hub import HfApi
from IPython.display import display
from transformers import AutoTokenizer
from common import read_rows
from configure_chat import configure_tokenizer
from prepare_data import accept, normalize, to_messages
from svg_utils import SYSTEM_PROMPT, extract_svg, render_svg
from validate_dataset import validate

DATASET_ID = 'JackertheHacker/svg-stack-qwen-captioned-subset'
CAPS = {'eval': 60, 'valid': 400, 'train': 6000}
SOURCE_SPLITS = {'eval': 'test', 'valid': 'validation', 'train': 'train'}
MAX_SVG_TOKENS = 1800
BUDGET_SECONDS = 4 * 60 * 60
config = yaml.safe_load((ROOT / 'motifforge/config.yaml').read_text())
model_entry = yaml.safe_load((ROOT / 'configs/models.yaml').read_text())['models'][config['model_alias']]
MODEL = {'id': model_entry['id'], 'revision': model_entry['revision']}
SPEC = {
    'config': config, 'model': MODEL, 'repo_revision': repo_revision, 'dataset': DATASET_ID,
    'caps': CAPS, 'source_splits': SOURCE_SPLITS, 'max_svg_tokens': MAX_SVG_TOKENS,
    'time_budget_seconds': BUDGET_SECONDS, 'quantization_for_training': '4bit',
    'optimizer': 'adamw_8bit', 'warmup_ratio': 0.1, 'packing': False,
    'generation': {'max_new_tokens': config['max_new_tokens'], 'do_sample': False},
}
config_path = RUN / 'settings.json'
if config_path.exists():
    if read_json(config_path) != SPEC:
        raise ValueError('Settings changed since this run started. Restore them or start a new run.')
else:
    save_json(config_path, SPEC)

data_tokenizer = configure_tokenizer(AutoTokenizer.from_pretrained(MODEL['id'], revision=MODEL['revision']))
project_path = ROOT / 'motifforge/eval_prompts_motifs.jsonl'
project_rows = read_rows(project_path)
if len(project_rows) != 20:
    raise ValueError('The project evaluation file must contain 20 prompts.')

DATA = RUN / 'data'
DATA.mkdir(exist_ok=True)
candidate_manifest = DATA / 'candidates.json'
dataset_manifest = DATA / 'manifest.json'
lock_file = RUN / 'training-plan.json'

def references_problem(svg):
    # Permit only local fragment references. Also block inline event handlers and CSS escapes.
    if re.search(r'<!DOCTYPE|<!ENTITY|@import|\\', svg, re.IGNORECASE):
        return 'external or unsupported SVG reference'
    try:
        root = ET.fromstring(svg)
    except ET.ParseError:
        return None  # The normal SVG validator reports the XML error.
    for element in root.iter():
        for attribute, value in element.attrib.items():
            name = attribute.rsplit('}', 1)[-1].lower()
            if name.startswith('on') or (name == 'href' and not value.strip().startswith('#')):
                return 'external reference or event handler'
        for value in [*element.attrib.values(), element.text or '']:
            for target in re.findall(r'url\s*\(([^)]*)\)', value, re.IGNORECASE):
                if not target.strip().strip("'\x22").strip().startswith('#'):
                    return 'external SVG URL'
    return None

def chat_length(tokenizer, messages):
    return len(tokenizer.apply_chat_template(
        messages, tokenize=True, return_dict=False, enable_thinking=False,
    ))

def collect_rows(stream, target, seen_prompts, seen_svgs):
    kept, reasons = [], Counter()
    for scanned, row in enumerate(stream, 1):
        item, reason = accept(row, data_tokenizer, MAX_SVG_TOKENS)
        if item:
            example = to_messages(item['caption'], item['svg'])
            prompt_key = normalize(item['caption'])
            svg_key = hashlib.sha256(item['svg'].encode()).hexdigest()
            reason = references_problem(item['svg'])
            if prompt_key in seen_prompts or svg_key in seen_svgs:
                reason = 'duplicate caption or SVG'
            elif not reason and chat_length(data_tokenizer, example['messages']) > config['max_seq_length']:
                reason = 'complete chat too long'
            if not reason:
                seen_prompts.add(prompt_key)
                seen_svgs.add(svg_key)
                kept.append(example)
        if reason:
            reasons['malformed XML' if reason.startswith('xml:') else reason] += 1
        if len(kept) >= target or scanned >= target * 25:
            break
    return kept, dict(reasons)

def write_rows(path, rows):
    temporary = path.with_suffix('.jsonl.partial')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')
    temporary.replace(path)

def validate_saved_splits(manifest, paths):
    errors = []
    all_prompts = {normalize(row['messages'][0]['content']) for row in project_rows}
    all_svgs = set()
    counts = {}
    for name, path in paths.items():
        if not path.is_file() or digest(path) != manifest['sha256'][name]:
            raise ValueError(f'{path.name} is missing or changed. Restore the original file or start a new run.')
        issues, lengths, _ = validate(path, data_tokenizer, config['max_seq_length'])
        errors.extend(issues)
        counts[name] = len(lengths)
        for row in read_rows(path):
            _, user, assistant = row['messages']
            prompt = normalize(user['content'])
            svg_hash = hashlib.sha256(assistant['content'].encode()).hexdigest()
            if prompt in all_prompts or svg_hash in all_svgs:
                errors.append(f'Duplicate caption or SVG in {path.name}')
            all_prompts.add(prompt)
            all_svgs.add(svg_hash)
    if errors:
        raise ValueError('\n'.join(errors[:10]))
    if counts != manifest['splits'] or any(counts[split] != CAPS[split] for split in ('valid', 'eval')):
        raise ValueError(f'Unexpected split counts: {counts}. Training requires both held-out splits.')
    return counts

paths = {'eval': DATA / 'eval.jsonl', 'valid': DATA / 'valid.jsonl',
         'train': DATA / ('train.jsonl' if lock_file.exists() else 'train-candidates.jsonl')}
if lock_file.exists() or candidate_manifest.exists():
    file = dataset_manifest if lock_file.exists() else candidate_manifest
    manifest = read_json(file)
else:
    revision_path = DATA / 'source.json'
    if revision_path.exists():
        source_revision = read_json(revision_path)['revision']
    else:
        source_revision = HfApi().dataset_info(DATASET_ID).sha
        if not source_revision:
            raise RuntimeError('Could not resolve the dataset revision.')
        save_json(revision_path, {'revision': source_revision})
    manifest = {
        'dataset': DATASET_ID, 'revision': source_revision,
        'source_url': f'https://huggingface.co/datasets/{DATASET_ID}',
        'upstream_url': 'https://huggingface.co/datasets/starvector/svg-stack',
        'terms_note': 'Ahmad confirmed both sources for the project. Credit StarVector and the captioned subset.',
        'seed': config['seed'], 'shuffle_buffer': 1000, 'source_splits': SOURCE_SPLITS,
        'max_svg_tokens': MAX_SVG_TOKENS, 'max_chat_tokens': config['max_seq_length'],
        'tokenizer': MODEL, 'system_prompt': SYSTEM_PROMPT, 'project_prompts_sha256': digest(project_path),
        'created_at': now(), 'splits': {}, 'rejected': {}, 'sha256': {},
    }
    seen_prompts = {normalize(row['messages'][0]['content']) for row in project_rows}
    seen_svgs = set()
    for name, source_split in SOURCE_SPLITS.items():
        stream = load_dataset(DATASET_ID, split=source_split, revision=source_revision, streaming=True)
        rows, rejected = collect_rows(stream.shuffle(seed=config['seed'], buffer_size=1000),
                                      CAPS[name], seen_prompts, seen_svgs)
        print(name, 'kept:', len(rows), '| rejected:', rejected)
        if name != 'train' and len(rows) != CAPS[name]:
            raise ValueError(f'{name} requires {CAPS[name]} usable examples. Training has not started.')
        write_rows(paths[name], rows)
        manifest['splits'][name] = len(rows)
        manifest['rejected'][name] = rejected
        manifest['sha256'][name] = digest(paths[name])
    save_json(candidate_manifest, manifest)

if manifest['project_prompts_sha256'] != digest(project_path):
    raise ValueError('The separate project prompts changed. Restore the original prompts or start a new run.')
counts = validate_saved_splits(manifest, paths)
if counts['train'] < config['gradient_accumulation_steps']:
    raise ValueError('Too few usable training rows for a full optimizer step.')
if lock_file.exists() and read_json(lock_file)['dataset_manifest_sha256'] != digest(dataset_manifest):
    raise ValueError('The saved dataset manifest changed. Restore it or start a new run.')
print('Ready:', counts)
print('Rejections by split:', manifest['rejected'])
for row in read_rows(paths['train'])[:4]:
    print(row['messages'][1]['content'])
    preview = render_svg(row['messages'][2]['content'], size=192)
    if preview is not None:
        display(preview)
    else:
        print('This sample could not be rendered.')

## 4. Load the model and estimate the run size

This cell measures speed and chooses how many training examples fit the four-hour estimate. It uses the main dataset and leaves the model weights untouched. The chosen split is saved once and reused on resume. The four hours are a planning target. If the run takes longer, let both epochs finish or resume them from Drive.

In [ ]:
import math
import statistics
import time

from peft.utils.save_and_load import load_peft_weights, set_peft_model_state_dict
from transformers import set_seed

set_seed(config['seed'])
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL['id'], revision=MODEL['revision'], use_exact_model_name=True,
    max_seq_length=config['max_seq_length'], load_in_4bit=True,
)
if not getattr(model, 'is_loaded_in_4bit', False):
    raise RuntimeError('The model was not loaded in 4 bit. Check the Unsloth install and GPU before continuing.')
configure_tokenizer(tokenizer)
model = FastLanguageModel.get_peft_model(
    model, r=config['rank'], lora_alpha=config['alpha'], lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=config['seed'],
)
model.print_trainable_parameters()
GPU = {'name': torch.cuda.get_device_name(0), 'vram_bytes': torch.cuda.get_device_properties(0).total_memory}
VERSIONS = {package: version(package) for package in
            ('torch', 'unsloth', 'unsloth_zoo', 'transformers', 'trl', 'peft', 'bitsandbytes', 'datasets')}
print(GPU, VERSIONS)

def encode(row):
    ids = tokenizer.apply_chat_template(row['messages'], tokenize=True, return_dict=False, enable_thinking=False)
    if len(ids) > config['max_seq_length']:
        raise ValueError('A row exceeds the model tokenizer limit. Check the data before training.')
    return {'input_ids': ids, 'attention_mask': [1] * len(ids)}

def generate(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)
    inputs = tokenizer(text, return_tensors='pt', add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, **SPEC['generation'], pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

evaluation = [
    {'group': 'held_out', 'id': f'test-{index:03d}', 'messages': row['messages'][:-1],
     'prompt': row['messages'][1]['content'], 'reference_svg': row['messages'][-1]['content']}
    for index, row in enumerate(read_rows(paths['eval']), 1)
]
evaluation += [
    {'group': 'project_prompts', 'id': row['id'],
     'messages': [{'role': 'system', 'content': SYSTEM_PROMPT}, *row['messages']],
     'prompt': row['messages'][-1]['content'], 'reference_svg': ''}
    for row in project_rows
]
if len({item['id'] for item in evaluation}) != len(evaluation):
    raise ValueError('Evaluation IDs must be unique.')

baseline_file = RUN / 'base-answers.json'
tuned_file = RUN / 'tuned-answers.json'

def collect_answers(path, items):
    answers = read_json(path) if path.exists() else {}
    for index, item in enumerate(items, 1):
        if item['id'] not in answers:
            started = time.monotonic()
            text = generate(item['messages'])
            answers[item['id']] = {'text': text, 'seconds': time.monotonic() - started}
            save_json(path, answers)
        if index % 10 == 0 or index == len(items):
            print(path.stem, index, '/', len(items))
    return answers

def measure_loss(records, count, backward):
    ordered = sorted(records, key=lambda row: len(row['input_ids']))
    selected = [ordered[round(index * (len(ordered) - 1) / (count - 1))] for index in range(count)]
    times = []
    model.train(backward)
    for index, record in enumerate([ordered[len(ordered) // 2], *selected]):
        inputs = {key: torch.tensor([value], device=model.device) for key, value in record.items()}
        torch.cuda.synchronize()
        started = time.monotonic()
        with torch.set_grad_enabled(backward), torch.autocast('cuda', dtype=model.dtype):
            loss = model(**inputs, labels=inputs['input_ids'], use_cache=False).loss
        if backward:
            loss.backward()
            model.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        if index:
            times.append(time.monotonic() - started)
    return statistics.mean(times)

def choose_train_count(available, accumulation, epochs, seconds, train_row, valid_row, valid_count,
                       generation_row, prompt_count):
    reserved = 1.25 * (epochs * valid_count * valid_row + 2 * prompt_count * generation_row)
    affordable = math.floor((seconds - reserved) / (epochs * 1.20 * train_row))
    return max(0, min(available, affordable) // accumulation * accumulation)

if lock_file.exists():
    training_plan = read_json(lock_file)
else:
    candidates = read_rows(paths['train'])
    calibration_train = [encode(row) for row in candidates]
    calibration_valid = [encode(row) for row in read_rows(paths['valid'])]
    FastLanguageModel.for_training(model)
    train_row_seconds = measure_loss(calibration_train, count=16, backward=True)
    valid_row_seconds = measure_loss(calibration_valid, count=8, backward=False)
    del calibration_train, calibration_valid
    FastLanguageModel.for_inference(model)
    sample = evaluation[::10]
    with model.disable_adapter():
        collect_answers(baseline_file, sample[:1])  # Warm up generation before taking the estimate.
        sample_answers = collect_answers(baseline_file, sample[1:])
    generation_row_seconds = statistics.mean(sample_answers[item['id']]['seconds'] for item in sample[1:])
    accumulation = config['gradient_accumulation_steps']
    n_train = choose_train_count(len(candidates), accumulation, config['epochs'], BUDGET_SECONDS,
                                 train_row_seconds, valid_row_seconds, CAPS['valid'],
                                 generation_row_seconds, len(evaluation))
    if not n_train:
        raise RuntimeError('The four-hour estimate cannot fit one training step. Recheck GPU speed before proceeding.')
    paths['train'] = DATA / 'train.jsonl'
    write_rows(paths['train'], candidates[:n_train])
    del candidates
    manifest['candidate_train_count'] = manifest['splits']['train']
    manifest['splits']['train'] = n_train
    manifest['sha256']['train'] = digest(paths['train'])
    save_json(dataset_manifest, manifest)
    validate_saved_splits(manifest, paths)
    estimated = (1.20 * config['epochs'] * n_train * train_row_seconds +
                 1.25 * (config['epochs'] * CAPS['valid'] * valid_row_seconds +
                         2 * len(evaluation) * generation_row_seconds))
    training_plan = {
        'train_count': n_train, 'estimated_seconds': round(estimated),
        'train_row_seconds': train_row_seconds, 'valid_row_seconds': valid_row_seconds,
        'generation_row_seconds': generation_row_seconds,
        'save_steps': max(1, round(900 / (train_row_seconds * accumulation))),
        'dataset_manifest_sha256': digest(dataset_manifest), 'sized_on': GPU,
    }
    save_json(lock_file, training_plan)
    (DATA / 'train-candidates.jsonl').unlink()
print('Train examples:', training_plan['train_count'])
print('Estimated hours including validation and evaluation:', round(training_plan['estimated_seconds'] / 3600, 2))
print('Save a resumable checkpoint every', training_plan['save_steps'], 'optimizer steps.')

## 5. Save the base answers

The base model sees the same prompts and generation settings as the tuned model. Each result is saved to Drive as it finishes. The 60 held-out targets stay out of the prompts and the trainer.

In [ ]:
FastLanguageModel.for_inference(model)
with model.disable_adapter():
    base_answers = collect_answers(baseline_file, evaluation)

## 6. Train for two epochs

Check the data preview and row count before running this cell. Watch for a finite training loss and validation loss. On a new session, the notebook restores the newest complete checkpoint for this run. Each save includes the optimizer, so training can continue where it left off.

In [ ]:
import tempfile
from zipfile import BadZipFile, ZIP_DEFLATED, ZipFile

from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer

CHECKPOINTS = RUN / 'checkpoints'
CHECKPOINTS.mkdir(exist_ok=True)
for interrupted in CHECKPOINTS.glob('checkpoint-*.zip.partial'):
    interrupted.unlink()
ADAPTER = RUN / 'adapter.zip'
COMPLETED = RUN / 'training-complete.json'

def pack_folder(source, destination):
    temporary = destination.with_suffix('.zip.partial')
    with ZipFile(temporary, 'w', ZIP_DEFLATED, compresslevel=1) as archive:
        for file in sorted(source.rglob('*')):
            if file.is_file():
                archive.write(file, file.relative_to(source))
    temporary.replace(destination)

def checkpoint_files():
    files = (path for path in CHECKPOINTS.glob('checkpoint-*.zip')
             if re.fullmatch(r'checkpoint-\d+\.zip', path.name))
    return sorted(files, key=lambda path: int(path.stem.split('-')[-1]), reverse=True)

def checkpoint_is_complete(folder, step):
    required = ('trainer_state.json', 'adapter_config.json', 'adapter_model.safetensors',
                'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'motifforge-progress.json')
    return (all((folder / file).is_file() for file in required)
            and read_json(folder / 'trainer_state.json')['global_step'] == step
            and read_json(folder / 'motifforge-progress.json')['plan_sha256'] == digest(lock_file))

def restore_checkpoint(root):
    for path in checkpoint_files():
        unpack = root / path.stem
        try:
            shutil.unpack_archive(path, unpack)
            step = int(path.stem.split('-')[-1])
            if checkpoint_is_complete(unpack, step):
                print('Resuming from', path.name)
                return unpack
        except (ValueError, OSError, shutil.ReadError, BadZipFile, KeyError) as exc:
            print('Skipping unreadable checkpoint', path.name, str(exc))
        shutil.rmtree(unpack, ignore_errors=True)
    if checkpoint_files():
        raise RuntimeError('No saved checkpoint is usable for these settings. Restore a checkpoint or start a new run.')
    return None

class SaveToDrive(TrainerCallback):
    def __init__(self, started, prior):
        self.started = started
        self.prior = prior

    def on_save(self, args, state, control, **kwargs):
        folder = Path(args.output_dir) / f'checkpoint-{state.global_step}'
        progress = {
            'plan_sha256': digest(lock_file),
            'train_seconds': self.prior['train_seconds'] + time.monotonic() - self.started,
            'peak_vram_bytes': max(self.prior['peak_vram_bytes'], torch.cuda.max_memory_allocated()),
            'sessions': self.prior['sessions'] + [{'gpu': GPU, 'versions': VERSIONS}],
        }
        save_json(folder / 'motifforge-progress.json', progress)
        if not checkpoint_is_complete(folder, state.global_step):
            raise RuntimeError(f'Checkpoint {folder.name} is missing optimizer or adapter state.')
        saved = CHECKPOINTS / (folder.name + '.zip')
        pack_folder(folder, saved)
        for old in checkpoint_files()[2:]:
            old.unlink()
        print('Saved to Drive:', saved.name)

if COMPLETED.exists():
    if not ADAPTER.is_file() or read_json(COMPLETED)['adapter_sha256'] != digest(ADAPTER):
        raise ValueError('The saved adapter is missing or changed. Restore adapter.zip before evaluating.')
    print('Both epochs already finished. Continue to evaluation.')
else:
    local_checkpoints = Path(tempfile.mkdtemp(prefix='trainer-', dir=WORK))
    resume_from = restore_checkpoint(local_checkpoints)
    prior = (read_json(resume_from / 'motifforge-progress.json') if resume_from else
             {'train_seconds': 0, 'peak_vram_bytes': 0, 'sessions': []})
    train_data = Dataset.from_list([encode(row) for row in read_rows(paths['train'])])
    valid_data = Dataset.from_list([encode(row) for row in read_rows(paths['valid'])])
    bf16 = is_bfloat16_supported()
    trainer = SFTTrainer(
        model=model, processing_class=tokenizer, train_dataset=train_data, eval_dataset=valid_data,
        args=SFTConfig(
            output_dir=str(local_checkpoints), max_length=config['max_seq_length'],
            per_device_train_batch_size=config['batch_size'], per_device_eval_batch_size=1,
            gradient_accumulation_steps=config['gradient_accumulation_steps'],
            learning_rate=config['learning_rate'], num_train_epochs=config['epochs'],
            warmup_ratio=SPEC['warmup_ratio'], optim=SPEC['optimizer'], fp16=not bf16, bf16=bf16,
            eval_strategy='epoch', save_strategy='steps', save_steps=training_plan['save_steps'],
            save_total_limit=2, save_safetensors=True, logging_steps=10,
            seed=config['seed'], data_seed=config['seed'], report_to='none', packing=False,
            padding_free=False,
        ),
    )
    FastLanguageModel.for_training(model)
    torch.cuda.reset_peak_memory_stats()
    training_started = time.monotonic()
    trainer.add_callback(SaveToDrive(training_started, prior))
    trainer.train(resume_from_checkpoint=str(resume_from) if resume_from else None)
    if trainer.state.epoch is None or trainer.state.epoch < config['epochs'] - 1e-6:
        raise RuntimeError('Training ended before both epochs completed. Resume this run before evaluation.')

    with tempfile.TemporaryDirectory(prefix='adapter-', dir=WORK) as folder:
        model.save_pretrained(folder)
        tokenizer.save_pretrained(folder)
        pack_folder(Path(folder), ADAPTER)
    save_json(RUN / 'training-log.json', trainer.state.log_history)
    save_json(COMPLETED, {
        'finished_at': now(), 'epochs': trainer.state.epoch, 'optimizer_steps': trainer.state.global_step,
        'train_seconds': prior['train_seconds'] + time.monotonic() - training_started,
        'peak_vram_bytes': max(prior['peak_vram_bytes'], torch.cuda.max_memory_allocated()),
        'sessions': prior['sessions'] + [{'gpu': GPU, 'versions': VERSIONS}],
        'adapter_sha256': digest(ADAPTER), 'plan_sha256': digest(lock_file),
    })
    print('Adapter saved:', ADAPTER)
    del trainer

## 7. Score the tuned model and save the evidence

The report shows the 60 test examples and the 20 project prompts separately. Review side-by-side images and failed generations. The team can use the examples to rate prompt adherence and visual quality on a 1–5 scale.

In [ ]:
from evaluate import score_svg, summarize, write_images, report_markdown

def load_saved_adapter():
    if not COMPLETED.is_file():
        raise RuntimeError('Finish training before evaluating or exporting.')
    completion = read_json(COMPLETED)
    if completion['plan_sha256'] != digest(lock_file) or completion['adapter_sha256'] != digest(ADAPTER):
        raise ValueError('The adapter or training plan changed. Restore the original run files.')
    with tempfile.TemporaryDirectory(prefix='restore-adapter-', dir=WORK) as folder:
        shutil.unpack_archive(ADAPTER, folder)
        set_peft_model_state_dict(model, load_peft_weights(folder, device='cpu'))
    return completion

completion = load_saved_adapter()
FastLanguageModel.for_inference(model)
tuned_answers = collect_answers(tuned_file, evaluation)
comparison = [
    {'group': item['group'], 'id': item['id'], 'prompt': item['prompt'], 'reference_svg': item['reference_svg'],
     'base_svg': base_answers[item['id']]['text'], 'tuned_svg': tuned_answers[item['id']]['text']}
    for item in evaluation
]
save_json(RUN / 'comparison.json', comparison)

def safe_score(svg, reference):
    clean = extract_svg(svg) or svg
    problem = references_problem(clean)
    if problem:
        return {**score_svg(''), 'chars': len(clean), 'reason': problem}
    return score_svg(svg, reference)

summary = {}
evaluation_dir = RUN / 'evaluation'
for group in ('held_out', 'project_prompts'):
    folder = evaluation_dir / group
    folder.mkdir(parents=True, exist_ok=True)
    group_records = [row for row in comparison if row['group'] == group]
    scored = []
    for index, record in enumerate(group_records, 1):
        reference = record['reference_svg']
        base = safe_score(record['base_svg'], reference)
        tuned = safe_score(record['tuned_svg'], reference)
        scored.append({'id': record['id'], 'prompt': record['prompt'], 'base': base, 'tuned': tuned})
        write_images(folder, index, {
            'reference_svg': reference,
            'base_svg': record['base_svg'] if base['render'] else '',
            'tuned_svg': record['tuned_svg'] if tuned['render'] else '',
        }, size=256)
    summary[group] = {'n': len(scored), 'size': 256,
                      'base': summarize(scored, 'base'), 'tuned': summarize(scored, 'tuned')}
    save_json(folder / 'scored.json', scored)
    (folder / 'report.md').write_text(report_markdown(summary[group], scored), encoding='utf-8')
save_json(RUN / 'score_summary.json', summary)
save_json(RUN / 'run.json', {
    'run_name': RUN_NAME, 'settings': SPEC, 'training_plan': training_plan, 'training': completion,
    'dataset_splits': manifest['splits'], 'evaluated_at': now(),
    'evaluation_gpu': GPU, 'evaluation_versions': VERSIONS,
    'base_generation_seconds': sum(answer['seconds'] for answer in base_answers.values()),
    'tuned_generation_seconds': sum(answer['seconds'] for answer in tuned_answers.values()),
})
EVIDENCE = RUN / 'motifforge-evidence.zip'
main_files = ('run.json', 'score_summary.json', 'comparison.json', 'training-log.json', 'data/manifest.json')
temporary = EVIDENCE.with_suffix('.zip.partial')
with ZipFile(temporary, 'w', ZIP_DEFLATED) as archive:
    for name in main_files:
        archive.write(RUN / name, name)
    for file in sorted(evaluation_dir.rglob('*')):
        if file.is_file():
            archive.write(file, file.relative_to(RUN))
temporary.replace(EVIDENCE)
print(json.dumps(summary, indent=2))
print('Evidence:', EVIDENCE)
print('Adapter:', ADAPTER)

## 8. Download the adapter and evidence

Both files are already in Drive. Select a file and run the cell to download it, or retrieve it from Drive later.

In [ ]:
from google.colab import files

DOWNLOAD = 'skip' # @param ["skip", "evidence", "adapter"]
if DOWNLOAD != 'skip':
    artifact = {'evidence': RUN / 'motifforge-evidence.zip', 'adapter': RUN / 'adapter.zip'}[DOWNLOAD]
    if not artifact.is_file():
        raise FileNotFoundError('Finish the earlier cells first.')
    files.download(str(artifact))

## 9. Optional · Export Q4_K_M for Ollama

Tick **EXPORT_GGUF** after you have the adapter and evidence. Conversion uses temporary Colab disk space. Download the GGUF when it finishes. To save a second copy to Drive, first check that your quota has room for the file size printed below.

In [ ]:
EXPORT_GGUF = False # @param {type:"boolean"}
gguf = None
if EXPORT_GGUF:
    if not (RUN / 'motifforge-evidence.zip').is_file():
        raise RuntimeError('Finish and save the evaluation before exporting.')
    load_saved_adapter()
    export_dir = Path(tempfile.mkdtemp(prefix='gguf-', dir=WORK))
    model.save_pretrained_gguf(str(export_dir / 'motifforge'), tokenizer,
                               quantization_method=config['quantization'])
    exports = [path for path in export_dir.rglob('*.gguf') if 'q4_k_m' in path.name.lower()]
    if len(exports) != 1 or not exports[0].stat().st_size:
        raise RuntimeError('Expected one Q4_K_M GGUF. Check the export output above.')
    gguf = exports[0]
    print(gguf.name, '| bytes:', gguf.stat().st_size)
    save_json(RUN / 'export.json', {'filename': gguf.name, 'bytes': gguf.stat().st_size,
                                  'quantization': config['quantization'], 'finished_at': now()})

In [ ]:
GGUF_DESTINATION = 'download' # @param ["download", "drive"]
if gguf is None:
    print('Enable and run the export cell when you want to create the GGUF.')
elif GGUF_DESTINATION == 'drive':
    destination = RUN / gguf.name
    temporary = destination.with_suffix('.gguf.partial')
    shutil.copyfile(gguf, temporary)
    temporary.replace(destination)
    print('Saved:', destination)
else:
    from google.colab import files

    files.download(str(gguf))